In [0]:
# Cell 1 — Widget
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run "../utils/config_loader"

In [0]:
# Cell 3 — Load config
config = load_config_from_widget()
print(f"🚀 DQ Pharmacy Pipeline - {config['environment'].upper()} Environment")
print(f"Status: Running data quality checks...")

# Cell 4 — Imports
from pyspark.sql.functions import *

# Cell 5 — Load tables
bronze_table = f"{config['catalog']}.{config['schemas']['bronze']}.pharmacy_raw"
silver_table = f"{config['catalog']}.{config['schemas']['silver']}.pharmacy_clean"
fact_table   = f"{config['catalog']}.{config['schemas']['gold']}.fact_sales"
dim_stores   = f"{config['catalog']}.{config['schemas']['gold']}.dim_stores"
dim_drugs    = f"{config['catalog']}.{config['schemas']['gold']}.dim_drugs"

df_bronze = spark.table(bronze_table)
df_silver = spark.table(silver_table)
df_fact   = spark.table(fact_table)
df_stores = spark.table(dim_stores)
df_drugs  = spark.table(dim_drugs)

# Cell 6 — DQ Checks
print("=== DATA QUALITY CHECKS ===\n")

# Check 1 — Bronze row count
bronze_count = df_bronze.count()
status = "PASS ✅" if 10 <= bronze_count <= 10000 else "FAIL ❌"
print(f"[BRONZE] Row count ({bronze_count})          : {status}")

# Check 2 — No nulls in sale_id
nulls = df_bronze.filter(col("sale_id").isNull()).count()
status = "PASS ✅" if nulls == 0 else "FAIL ❌"
print(f"[BRONZE] No nulls in sale_id               : {status}")


# Check 3 — Silver retention rate
silver_count = df_silver.count()
retention = (silver_count / bronze_count) * 100
retention = int(retention * 100) / 100
status = "PASS ✅" if retention >= 50 else "FAIL ❌"
print(f"[SILVER] Retention rate ({retention} percent) : {status}")

# Check 4 — No nulls in total_amount
nulls = df_silver.filter(col("total_amount").isNull()).count()
status = "PASS ✅" if nulls == 0 else "FAIL ❌"
print(f"[SILVER] No nulls in total_amount          : {status}")

# Check 5 — No negative amounts
neg = df_silver.filter(col("total_amount") < 0).count()
status = "PASS ✅" if neg == 0 else "FAIL ❌"
print(f"[SILVER] No negative amounts               : {status}")

# Check 6 — Valid sale_priority values
valid_priorities = ["Critical", "High", "Normal"]
invalid = df_silver.filter(~col("sale_priority").isin(valid_priorities)).count()
status = "PASS ✅" if invalid == 0 else "FAIL ❌"
print(f"[SILVER] Valid sale_priority values        : {status}")

# Check 7 — fact_sales matches silver
fact_count = df_fact.count()
status = "PASS ✅" if fact_count == silver_count else "FAIL ❌"
print(f"[GOLD]   fact_sales matches silver         : {status}")

# Check 8 — No orphan stores in fact
orphan_stores = df_fact.join(df_stores, on="store_id", how="left_anti").count()
status = "PASS ✅" if orphan_stores == 0 else "FAIL ❌"
print(f"[GOLD]   No orphan stores in fact_sales   : {status}")

# Check 9 — No orphan drugs in fact
orphan_drugs = df_fact.join(df_drugs, on="drug_id", how="left_anti").count()
status = "PASS ✅" if orphan_drugs == 0 else "FAIL ❌"
print(f"[GOLD]   No orphan drugs in fact_sales    : {status}")

# Check 10 — dim_stores has data
status = "PASS ✅" if df_stores.count() > 0 else "FAIL ❌"
print(f"[GOLD]   dim_stores has data              : {status}")

# Cell 7 — Summary
print(f"\n=== DQ SUMMARY ===")
print(f"🌍 Environment  : {config['environment'].upper()}")
print(f"📥 Bronze rows  : {bronze_count}")
print(f"✅ Silver rows  : {silver_count}")
print(f"📊 Retention    : {retention}%")
print(f"✅ fact_sales   : {fact_count}")